# BleedCheck AI Arm: Iron Deficiency in Women and Girls (NHANES 2017-2020)
Heavy periods drain iron. This model predicts iron deficiency (ferritin < 15 ng/mL, WHO cutoff) from a routine
CBC in non-pregnant females aged 12-49, the exact group NHANES tested for ferritin.
Ferritin builds the label and is never a feature (leakage guard below).

**Files needed in Colab `sample_data/`:** P_DEMO.xpt, P_CBC.xpt, P_FERTIN.xpt, P_HSCRP.xpt (same as CBCIron).
Then Runtime > Run all.

In [1]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
RNG = 42
P = "sample_data/"
FERRITIN_CUTOFF = 15   # WHO iron deficiency cutoff for age > 5. Change to 30 for a sensitivity analysis.

In [2]:
rd = lambda f: pd.read_sas(P + f, format="xport")
demo, cbc, fer, crp = rd("P_DEMO.xpt"), rd("P_CBC.xpt"), rd("P_FERTIN.xpt"), rd("P_HSCRP.xpt")
d = demo[["SEQN", "RIAGENDR", "RIDAGEYR", "RIDEXPRG"]].merge(cbc, on="SEQN").merge(fer[["SEQN", "LBXFER"]], on="SEQN").merge(crp[["SEQN", "LBXHSCRP"]], on="SEQN", how="left")
d = d[(d.RIAGENDR == 2) & d.RIDAGEYR.between(12, 49) & (d.RIDEXPRG != 1)]   # females 12-49, not pregnant
d = d.dropna(subset=["LBXFER", "LBXHGB", "LBXMCVSI", "LBXRDW"])
d["iron_deficient"] = (d.LBXFER < FERRITIN_CUTOFF).astype(int)
print(f"Cohort: {len(d)} females 12-49 | iron deficient: {d.iron_deficient.sum()} ({d.iron_deficient.mean():.1%})")
print(f"Iron deficient WITHOUT anemia (Hgb >= 12): {((d.iron_deficient == 1) & (d.LBXHGB >= 12)).sum()}")

Cohort: 2656 females 12-49 | iron deficient: 461 (17.4%)
Iron deficient WITHOUT anemia (Hgb >= 12): 221


In [3]:
CANDIDATES = ["RIDAGEYR", "LBXHGB", "LBXMCVSI", "LBXRDW", "LBXRBCSI", "LBXPLTSI", "LBXMC1SI", "LBXHSCRP"]
FEATURES = [c for c in CANDIDATES if c in d.columns]
assert not ({"LBXFER", "iron_deficient"} & set(FEATURES)), "LEAKAGE: ferritin/label in features"
d = d.dropna(subset=FEATURES)
X, y = d[FEATURES], d.iron_deficient.values
print("Leakage guard passed. Features:", FEATURES, "| n =", len(d))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)

Leakage guard passed. Features: ['RIDAGEYR', 'LBXHGB', 'LBXMCVSI', 'LBXRDW', 'LBXRBCSI', 'LBXPLTSI', 'LBXHSCRP'] | n = 2614


In [4]:
rf = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=5, class_weight="balanced", random_state=RNG)
lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=RNG)
for name, m in [("Random forest", rf), ("Logistic regression", lr)]:
    s = cross_val_score(m, Xtr, ytr, cv=cv, scoring="roc_auc")
    print(f"{name}: 5-fold CV AUC {s.mean():.3f} +/- {s.std():.3f}")
rf.fit(Xtr, ytr); p = rf.predict_proba(Xte)[:, 1]
print(f"\nTest AUC: {roc_auc_score(yte, p):.3f}")
print(classification_report(yte, (p >= 0.5).astype(int), target_names=["not ID", "iron deficient"]))
ysh = np.random.default_rng(RNG).permutation(ytr)
sh = cross_val_score(rf, Xtr, ysh, cv=cv, scoring="roc_auc").mean()
print(f"Shuffled-label control AUC: {sh:.3f}", "(OK)" if sh < 0.6 else "WARNING")

Random forest: 5-fold CV AUC 0.889 +/- 0.021
Logistic regression: 5-fold CV AUC 0.905 +/- 0.018



Test AUC: 0.882
                precision    recall  f1-score   support

        not ID       0.93      0.89      0.91       432
iron deficient       0.56      0.68      0.62        91

      accuracy                           0.85       523
     macro avg       0.75      0.79      0.76       523
  weighted avg       0.87      0.85      0.86       523



Shuffled-label control AUC: 0.492 (OK)


In [5]:
fpr, tpr, thr = roc_curve(yte, p)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fpr, tpr, label=f"AUC {roc_auc_score(yte, p):.2f}"); ax[0].plot([0, 1], [0, 1], "k:")
ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("Sensitivity"); ax[0].set_title("Iron deficiency, females 12-49"); ax[0].legend()
pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh(ax=ax[1], title="Feature importance")
plt.tight_layout(); plt.savefig("iron_results.png", dpi=150); plt.show()
print("Threshold table:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    tn, fp, fn, tp = confusion_matrix(yte, (p >= t).astype(int)).ravel()
    print(f"  {t:.1f}: sensitivity {tp/(tp+fn):.0%}  specificity {tn/(tn+fp):.0%}")

Threshold table:
  0.3: sensitivity 80%  specificity 77%
  0.4: sensitivity 78%  specificity 83%
  0.5: sensitivity 68%  specificity 89%
  0.6: sensitivity 62%  specificity 93%
  0.7: sensitivity 55%  specificity 95%


In [6]:
rf.fit(X, y)
joblib.dump(rf, "bleedcheck_iron_model.pkl")
card = {"name": "BleedCheck iron arm", "data": "NHANES 2017-March 2020", "cohort": "non-pregnant females 12-49",
        "label": f"ferritin < {FERRITIN_CUTOFF} ng/mL", "label_source_excluded": True, "features": FEATURES,
        "n": int(len(d)), "prevalence": round(float(y.mean()), 3), "test_auc": round(float(roc_auc_score(yte, p)), 3),
        "shuffled_auc": round(float(sh), 3), "survey_weights": "not used (prediction, not prevalence)",
        "not_for": "Diagnosis. High score = ask for a ferritin test."}
json.dump(card, open("model_card_iron.json", "w"), indent=2); print(json.dumps(card, indent=2))

{
  "name": "BleedCheck iron arm",
  "data": "NHANES 2017-March 2020",
  "cohort": "non-pregnant females 12-49",
  "label": "ferritin < 15 ng/mL",
  "label_source_excluded": true,
  "features": [
    "RIDAGEYR",
    "LBXHGB",
    "LBXMCVSI",
    "LBXRDW",
    "LBXRBCSI",
    "LBXPLTSI",
    "LBXHSCRP"
  ],
  "n": 2614,
  "prevalence": 0.174,
  "test_auc": 0.882,
  "shuffled_auc": 0.492,
  "survey_weights": "not used (prediction, not prevalence)",
  "not_for": "Diagnosis. High score = ask for a ferritin test."
}


## Export for the BleedCheck app
Logistic regression runs in any browser, so the app never sends lab values anywhere.
The cutoff is picked on training data only (cross-validated) to hit about 85% sensitivity, then checked on the untouched test set.
Copy the printed JSON into `IRON_MODEL` in bleedcheck.html, or send it to Claude to do it.

In [7]:
TARGET_SENS = 0.85
lr.fit(Xtr, ytr)
oof = cross_val_predict(lr, Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
f_, t_, th_ = roc_curve(ytr, oof); THR = float(th_[int(np.argmax(t_ >= TARGET_SENS))])
pl = lr.predict_proba(Xte)[:, 1]; tn, fp, fn, tp = confusion_matrix(yte, (pl >= THR).astype(int)).ravel()
print(f"Logistic regression test AUC {roc_auc_score(yte, pl):.3f} | cutoff {THR:.3f} | test sensitivity {tp/(tp+fn):.0%} specificity {tn/(tn+fp):.0%}")
sc_, clf_ = lr.named_steps["standardscaler"], lr.named_steps["logisticregression"]
app_model = {"features": FEATURES, "impute_medians": {f: float(Xtr[f].median()) for f in FEATURES},
             "scaler_mean": sc_.mean_.tolist(), "scaler_scale": sc_.scale_.tolist(),
             "coef": clf_.coef_[0].tolist(), "intercept": float(clf_.intercept_[0]), "threshold": THR,
             "sensitivity_at_threshold": round(tp/(tp+fn), 3), "specificity_at_threshold": round(tn/(tn+fp), 3),
             "test_auc": round(float(roc_auc_score(yte, pl)), 3), "n": int(len(d)),
             "label": f"ferritin < {FERRITIN_CUTOFF} ng/mL", "data": "NHANES 2017-March 2020, non-pregnant females 12-49"}
json.dump(app_model, open("bleedcheck_iron_model.json", "w"), indent=2)
print(json.dumps(app_model))
try:
    from google.colab import files; files.download("bleedcheck_iron_model.json"); files.download("iron_results.png")
except Exception: pass

Logistic regression test AUC 0.889 | cutoff 0.392 | test sensitivity 81% specificity 75%
{"features": ["RIDAGEYR", "LBXHGB", "LBXMCVSI", "LBXRDW", "LBXRBCSI", "LBXPLTSI", "LBXHSCRP"], "impute_medians": {"RIDAGEYR": 29.0, "LBXHGB": 13.2, "LBXMCVSI": 87.2, "LBXRDW": 13.5, "LBXRBCSI": 4.54, "LBXPLTSI": 269.0, "LBXHSCRP": 1.57}, "scaler_mean": [29.360114777618364, 13.083309421329508, 86.31468197034911, 13.940985174557627, 4.549292204686752, 276.0760401721664, 3.665160210425634], "scaler_scale": [11.386598749074768, 1.2606405078778544, 6.70006457150374, 1.641606562034516, 0.36524262781091565, 66.775543242484, 5.842910619345237], "coef": [-0.040787543932105264, -0.9696985743738447, -0.5000131146072079, 1.1612350841035974, -0.27154237166657635, 0.1655843320753038, -1.3052285818492544], "intercept": -0.8422250731270877, "threshold": 0.391734438802728, "sensitivity_at_threshold": 0.813, "specificity_at_threshold": 0.748, "test_auc": 0.889, "n": 2614, "label": "ferritin < 15 ng/mL", "data": "NHA